# Plant Disease Classification — ResNet9 (TensorFlow/Keras)

## 1. Imports & Setup

In [6]:
import pandas as pd
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, ReLU, MaxPool2D,
    GlobalAveragePooling2D, Flatten, Dense, Add, Dropout,
    RandomFlip, RandomRotation, RandomZoom, Rescaling
)
from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs available     : {tf.config.list_physical_devices("GPU")}')

# Use mixed precision for faster training on GPU (comment out if on CPU only)
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision: ON (float16)')
except Exception as e:
    print(f'Mixed precision not enabled: {e}')

TensorFlow version : 2.16.2
GPUs available     : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision: ON (float16)


## 2. Configuration

In [7]:
# ----------------------------------------------------------
# Paths — update DATA_DIR if your dataset is elsewhere
# ----------------------------------------------------------
DATA_DIR   = './New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
VALID_DIR  = os.path.join(DATA_DIR, 'valid')
TEST_DIR   = './test'

# ----------------------------------------------------------
# Hyper-parameters  (mirror of the PyTorch notebook)
# ----------------------------------------------------------
IMAGE_SIZE   = (128, 128)   # reduced for speed on Mac Metal
BATCH_SIZE   = 16           # reduced for speed on Mac Metal
NUM_CLASSES  = 38
EPOCHS       = 10
LEARNING_RATE= 0.01        # same starting LR as PyTorch (max_lr)
WEIGHT_DECAY = 1e-4
SEED         = 7

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('Configuration set. (128x128, Batch 16)')

Configuration set. (128x128, Batch 16)


## 3. Data Loading

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

training_set = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    interpolation='bilinear'
)

validation_set = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    interpolation='bilinear'
)

class_names = training_set.class_names
print(f'Classes found : {len(class_names)}')
print(f'Class names   : {class_names[:5]} ...')

Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Classes found : 38
Class names   : ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy'] ...


In [9]:
# Number of images for each disease
diseases = os.listdir(TRAIN_DIR)
nums = {}
for disease in diseases:
    nums[disease] = len(os.listdir(TRAIN_DIR + '/' + disease))
    
# converting the nums dictionary to pandas dataframe passing index as plant name and number of images as column

img_per_class = pd.DataFrame(nums.values(), index=nums.keys(), columns=["no. of images"])
img_per_class

NotADirectoryError: [Errno 20] Not a directory: './New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/.DS_Store'

## 4. Pipeline Optimisation

In [ ]:
# Cache + prefetch for GPU pipeline
training_set   = training_set.cache().prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.cache().prefetch(buffer_size=AUTOTUNE)

# Quick sanity-check — show one batch
for images, labels in training_set.take(1):
    print(f'Batch image shape : {images.shape}')
    print(f'Batch label shape : {labels.shape}')
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].numpy().astype('uint8'))
        ax.set_title(class_names[tf.argmax(labels[i]).numpy()], fontsize=8)
        ax.axis('off')
    plt.suptitle('Sample Training Images', fontsize=14)
    plt.tight_layout()
    plt.show()

## 5. ResNet9 Model (TF Functional API)

Exact mirror of the PyTorch `ResNet9` class:

```
conv1 → conv2(pool) → [res1] → conv3(pool) → conv4(pool) → [res2] → MaxPool → Flatten → Dense
```

In [ ]:
def conv_block(x, filters, pool=False, name_prefix=''):
    """
    Equivalent of PyTorch ConvBlock:
      Conv2D(kernel=3, pad='same') → BatchNorm → ReLU [ → MaxPool(4) ]
    """
    x = Conv2D(
        filters, kernel_size=3, padding='same', use_bias=False,
        name=f'{name_prefix}_conv'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_bn')(x)
    x = ReLU(name=f'{name_prefix}_relu')(x)
    if pool:
        x = MaxPool2D(pool_size=4, name=f'{name_prefix}_pool')(x)
    return x


def build_resnet9(num_classes=38, input_shape=(128, 128, 3)):
    """
    TF/Keras Functional API implementation of PyTorch ResNet9.
    *Lightweight Version* optimized for Mac MPS (1.6M parameters)
    """
    inputs = Input(shape=input_shape, name='input')

    # ── Data Augmentation (active during training only) ───────
    x = RandomFlip("horizontal_and_vertical")(inputs)
    x = RandomRotation(0.2)(x)
    x = RandomZoom(0.2)(x)

    # Normalise pixel values to [0, 1]
    x = Rescaling(1.0 / 255.0, name='rescaling')(x)

    # ── conv1 ───────────────────────────────────────────────
    x = conv_block(x, 32, name_prefix='conv1')           # (128, 128, 32)

    # ── conv2 (with MaxPool/4) ──────────────────────────────
    x = conv_block(x, 64, pool=True, name_prefix='conv2')  # (32, 32, 64)

    # ── res1 : two ConvBlocks + identity skip ───────────────
    skip1 = x
    x = conv_block(x, 64, name_prefix='res1a')          # (32, 32, 64)
    x = conv_block(x, 64, name_prefix='res1b')          # (32, 32, 64)
    x = Add(name='res1_add')([x, skip1])                 # residual connection

    # ── conv3 (with MaxPool/4) ───────────────────────────────
    x = conv_block(x, 128, pool=True, name_prefix='conv3')  # (8, 8, 128)

    # ── conv4 (with MaxPool/4) ───────────────────────────────
    x = conv_block(x, 256, pool=True, name_prefix='conv4')  # (2, 2, 256)

    # ── res2 : two ConvBlocks + identity skip ────────────────
    skip2 = x
    x = conv_block(x, 256, name_prefix='res2a')          # (2, 2, 256)
    x = conv_block(x, 256, name_prefix='res2b')          # (2, 2, 256)
    x = Add(name='res2_add')([x, skip2])                 # residual connection

    # ── classifier head ──────────────────────────────────────
    x = GlobalAveragePooling2D(name='classifier_pool')(x)   # (1, 1, 256) 
    # Use GlobalAveragePooling2D instead if pool drops below 4x4, but 2x2 with pool=2 works. 
    # Flatten is not needed after GlobalAveragePooling2D                           # (256,)
    outputs = Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

    model = Model(inputs, outputs, name='ResNet9_Light')
    return model


model = build_resnet9(num_classes=NUM_CLASSES)
model.summary(line_length=90)

## 6. Compile

In [ ]:
# AdamW mirrors PyTorch Adam + weight_decay
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compiled.')

## 7. Callbacks

| PyTorch | TF/Keras equivalent |
|---|---|
| `OneCycleLR` | `ReduceLROnPlateau` (reduce on plateau) |
| `grad_clip=0.1` | `clipvalue=0.1` in optimizer |
| manual best-model save | `ModelCheckpoint` |

In [ ]:
callbacks = [
    # Mirrors OneCycleLR: reduce LR when val_loss stops improving
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # Stop training if no improvement for 5 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Save best checkpoint
    tf.keras.callbacks.ModelCheckpoint(
        filepath='resnet9_plant_disease_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # TensorBoard logs (optional — run: tensorboard --logdir ./logs)
    tf.keras.callbacks.TensorBoard(log_dir='./logs', histogram_freq=1)
]
print('Callbacks ready.')

## 8. Train

In [ ]:
training_history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 9. Evaluate

In [ ]:
train_loss, train_acc = model.evaluate(training_set, verbose=0)
val_loss,   val_acc   = model.evaluate(validation_set, verbose=0)

print(f'Training   — Loss: {train_loss:.4f}  |  Accuracy: {train_acc*100:.2f}%')
print(f'Validation — Loss: {val_loss:.4f}  |  Accuracy: {val_acc*100:.2f}%')

## 10. Training History Plots

In [ ]:
hist = training_history.history
epochs_range = range(1, len(hist['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(epochs_range, hist['accuracy'],     '-ro', label='Training Accuracy')
axes[0].plot(epochs_range, hist['val_accuracy'], '-bo', label='Validation Accuracy')
axes[0].set_title('Accuracy vs Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss
axes[1].plot(epochs_range, hist['loss'],     '-rx', label='Training Loss')
axes[1].plot(epochs_range, hist['val_loss'], '-bx', label='Validation Loss')
axes[1].set_title('Loss vs Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.suptitle('ResNet9  —  Training History', fontsize=14)
plt.tight_layout()
plt.savefig('resnet9_training_history.png', dpi=150)
plt.show()

## 11. Learning Rate History

In [ ]:
if 'lr' in hist:
    plt.figure(figsize=(8, 4))
    plt.plot(epochs_range, hist['lr'], '-gs', label='Learning Rate')
    plt.title('Learning Rate Schedule (ReduceLROnPlateau)')
    plt.xlabel('Epoch'); plt.ylabel('LR')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12. Confusion Matrix & Classification Report

In [ ]:
# Rebuild validation set without caching / shuffling for ordered predictions
test_set = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=IMAGE_SIZE,
    batch_size=1,
    shuffle=False,
    interpolation='bilinear'
)

y_pred_probs = model.predict(test_set, verbose=1)
predicted_categories = tf.argmax(y_pred_probs, axis=1).numpy()

true_categories = tf.concat([tf.argmax(y, axis=1) for _, y in test_set], axis=0).numpy()

print('\n--- Classification Report ---')
print(classification_report(true_categories, predicted_categories, target_names=class_names))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(true_categories, predicted_categories)

plt.figure(figsize=(40, 40))
sns.heatmap(
    cm, annot=True, fmt='d', annot_kws={'size': 8},
    xticklabels=class_names, yticklabels=class_names,
    cmap='Blues'
)
plt.xlabel('Predicted Class', fontsize=20)
plt.ylabel('Actual Class', fontsize=20)
plt.title('Plant Disease Prediction — Confusion Matrix (ResNet9 TF)', fontsize=25)
plt.tight_layout()
plt.savefig('resnet9_confusion_matrix.png', dpi=100)
plt.show()

## 13. Predict on Single Test Image

In [ ]:
import cv2

def predict_image(image_path, model, class_names, image_size=IMAGE_SIZE):
    """
    Predict the disease class for a single image.
    Returns the class name and confidence score.
    """
    # Load & preprocess
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
    img_array   = tf.keras.preprocessing.image.img_to_array(img)
    img_batch   = tf.expand_dims(img_array, axis=0)   # shape is dynamic based on IMAGE_SIZE

    # Predict
    predictions   = model.predict(img_batch, verbose=0)
    predicted_idx = tf.argmax(predictions[0]).numpy()
    confidence    = float(predictions[0][predicted_idx]) * 100

    return class_names[predicted_idx], confidence


# ----- Run prediction on test images -----
test_image_dir = os.path.join(TEST_DIR, 'test') if os.path.isdir(os.path.join(TEST_DIR, 'test')) else TEST_DIR

test_images = sorted([
    f for f in os.listdir(test_image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f'Found {len(test_images)} test images in {test_image_dir}\n')

for img_name in test_images:
    img_path = os.path.join(test_image_dir, img_name)
    pred_class, confidence = predict_image(img_path, model, class_names)
    print(f'Image: {img_name:<40}  Predicted: {pred_class}  ({confidence:.1f}%)')

In [ ]:
# Visualise predictions for the first 8 test images
n_show = min(8, len(test_images))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    img_path = os.path.join(test_image_dir, test_images[i])
    pred_class, confidence = predict_image(img_path, model, class_names)

    # Display
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f'{pred_class}\n({confidence:.1f}%)', fontsize=8)
    ax.axis('off')

plt.suptitle('ResNet9 TF — Test Image Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('resnet9_test_predictions.png', dpi=150)
plt.show()

## 14. Save History & Model

In [ ]:
# Save training history to JSON
with open('resnet9_training_hist.json', 'w') as f:
    # Convert numpy values to Python floats for JSON serialisation
    serialisable_hist = {k: [float(v) for v in vals] for k, vals in hist.items()}
    json.dump(serialisable_hist, f, indent=2)
print('Training history saved → resnet9_training_hist.json')

# Save the final model
model.save('plant_disease_resnet9_tf.keras')
print('Model saved → plant_disease_resnet9_tf.keras')

## 15. Load Saved Model & Quick Inference Check

In [ ]:
# Verify the saved model loads and predicts correctly
loaded_model = tf.keras.models.load_model('plant_disease_resnet9_tf.keras')
print('Saved model loaded successfully.')
print(f'Classes: {loaded_model.output_shape}')

# Reload the class names (useful when running inference standalone)
val_ds_for_classes = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, image_size=IMAGE_SIZE, batch_size=1, shuffle=False
)
class_names_loaded = val_ds_for_classes.class_names
print(f'Loaded {len(class_names_loaded)} class names.')

# Test with first image if available
if test_images:
    sample_path = os.path.join(test_image_dir, test_images[0])
    pred_class, confidence = predict_image(sample_path, loaded_model, class_names_loaded)
    print(f'Sample prediction → {pred_class}  ({confidence:.1f}%)')